# 레슨 02 — 최종 미션 모범 답안

> 🔒 **교사·관리자 전용. 학생에게 배포 금지.**  
> 이 파일은 `final-mission.md` 의 기준 답안이다. 수치가 난수와 라이브러리 버전에 따라 아주 조금 달라질 수 있으므로, 채점은 정확한 숫자 암기보다 계산 흐름과 해석 일치 여부를 본다.

## 1. 환경 셀

In [ ]:
import os
import numpy as np

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ
if IS_COLAB:
    DATA_BASE = "https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-data-analysis/lectures/02/data"
else:
    DATA_BASE = "./data"
np.random.seed(42)
print("data base:", DATA_BASE)

## 2. 데이터 로드 및 구조 검증

In [ ]:
raw_stock = np.loadtxt(f"{DATA_BASE}/stock_prices.csv", delimiter=",", skiprows=1)
days = raw_stock[:, 0].astype(int)
prices = raw_stock[:, 1:]
names = np.array(["TechA", "FinB", "SmallC", "MidD", "LargeE"])

print("raw shape:", raw_stock.shape)
print("prices shape:", prices.shape)
print("first day:", days[0], "last day:", days[-1])

**채점 메모**: `raw_stock.shape` 는 `(252, 6)` 이고, 가격 배열 `prices.shape` 는 `(252, 5)` 가 되어야 한다. 학생이 `day` 열까지 가격에 포함하면 이후 수익률 계산이 왜곡된다.

## 3. 일간 로그수익률 계산

In [ ]:
log_ret = np.diff(np.log(prices), axis=0)
ret_mean = log_ret.mean(axis=0)
ret_std = log_ret.std(axis=0)

print("log_ret shape:", log_ret.shape)
for i, name in enumerate(names):
    print(f"{name:<7} 일평균 {ret_mean[i]*100:+.3f}% | 일변동성 {ret_std[i]*100:.3f}%")

`np.diff(..., axis=0)` 때문에 252일 가격이 251개 수익률로 바뀐다. `axis=1` 로 계산하면 종목끼리 차이를 내는 셈이라 오답이다.

## 4. 연환산 수익률과 변동성

In [ ]:
ann_ret = ret_mean * 252
ann_vol = ret_std * np.sqrt(252)

print(f"{'종목':<7} {'연수익률':>10} {'연변동성':>10}")
for i, name in enumerate(names):
    print(f"{name:<7} {ann_ret[i]*100:>9.1f}% {ann_vol[i]*100:>9.1f}%")

best_ret_idx = ann_ret.argmax()
best_vol_idx = ann_vol.argmax()
print("수익률 1위:", names[best_ret_idx])
print("변동성 1위:", names[best_vol_idx])

**해석 포인트**: 수익률이 높다고 무조건 좋은 종목은 아니다. 변동성이 같이 높으면 학생이 결론에서 리스크를 언급해야 한다.

## 5. z-점수 표준화와 극단값 탐색

In [ ]:
z = (log_ret - ret_mean) / ret_std
extreme_mask = np.abs(z) > 2.5
extreme_counts = extreme_mask.sum(axis=0)

print("표준화 후 평균:", z.mean(axis=0).round(4))
print("표준화 후 표준편차:", z.std(axis=0).round(4))
for i, name in enumerate(names):
    print(f"{name:<7} |z|>2.5 인 날: {int(extreme_counts[i])}일")

**채점 메모**: 표준화 후 평균은 거의 0, 표준편차는 거의 1이어야 한다. `axis` 를 빼고 전체 평균으로 표준화한 학생도 실행은 되지만 종목별 비교 목적에는 맞지 않으므로 감점한다.

## 6. 랜덤워크 시뮬레이션

In [ ]:
np.random.seed(2024)
n_sim = 1000
n_steps = 252

up_probs = []
final_price_samples = []

for i, name in enumerate(names):
    simulated_returns = np.random.normal(
        loc=ret_mean[i],
        scale=ret_std[i],
        size=(n_steps, n_sim),
    )
    simulated_paths = prices[0, i] * np.exp(np.cumsum(simulated_returns, axis=0))
    final_prices = simulated_paths[-1]
    up_prob = (final_prices > prices[0, i]).mean()
    up_probs.append(up_prob)
    final_price_samples.append(final_prices)
    print(f"{name:<7} 상승 확률: {up_prob*100:5.1f}% | 최종가 중앙값: {np.median(final_prices):,.0f}")

up_probs = np.array(up_probs)

시뮬레이션은 정답이 하나로 고정되는 계산이 아니다. 그래도 같은 seed 를 쓰면 학생과 강사의 수치가 거의 일치한다. seed 를 바꾼 학생은 결론과 코드 출력이 일치하면 통과 처리할 수 있다.

## 7. 보너스 B1 — 샤프 지수

In [ ]:
sharpe = ann_ret / ann_vol
for i, name in enumerate(names):
    print(f"{name:<7} Sharpe: {sharpe[i]:+.2f}")
print("위험 대비 효율 1위:", names[sharpe.argmax()])

연변동성이 0에 가까운 데이터라면 0으로 나누기 문제가 생기지만, 이 데이터는 모든 종목에 변동성을 넣어 두었다.

## 8. 보너스 B2 — 상관 행렬

In [ ]:
corr = np.corrcoef(log_ret.T)
print(corr.round(3))

upper = np.triu(np.ones_like(corr, dtype=bool), k=1)
pair_idx = np.where(upper)
best_pair_pos = corr[pair_idx].argmax()
i = pair_idx[0][best_pair_pos]
j = pair_idx[1][best_pair_pos]
print("상관이 가장 높은 쌍:", names[i], names[j], "r=", round(corr[i, j], 3))

상관 행렬의 대각선은 자기 자신과의 상관이라 항상 1이다. 자기 자신을 제외하려고 `np.triu(..., k=1)` 을 쓴다.

## 9. 보너스 B3 — 드로우다운

In [ ]:
for i, name in enumerate(names):
    running_peak = np.maximum.accumulate(prices[:, i])
    drawdown = (prices[:, i] - running_peak) / running_peak
    print(f"{name:<7} MDD: {drawdown.min()*100:.1f}%")

MDD 는 항상 0 이하의 값이다. 가장 작은 값이 최대 하락폭이다.

## 결론 예시

## 결론

1. 연환산 수익률 1위와 변동성 1위는 코드 출력 기준으로 다르거나 같을 수 있으므로, 수익률과 위험을 분리해 판단해야 한다.
2. z-점수 기준 |z|>2.5 인 날은 많지 않아 대부분의 일간 수익률은 평균 주변에 모여 있지만, 극단 손익일은 투자 리스크로 따로 관리해야 한다.
3. 랜덤워크 시뮬레이션에서 상승 확률이 50% 를 넘는 종목은 기대수익률이 양수인 후보로 볼 수 있다.
4. 단순 수익률만 보면 최고 수익률 종목을 고르겠지만, 위험 대비 효율까지 보려면 Sharpe 가 높은 종목을 우선 검토하겠다.

## 채점 기준 요약

| 항목 | 통과 기준 |
|---|---|
| 데이터 로드 | `day` 열과 가격 열을 분리했는가 |
| 로그수익률 | `np.diff(np.log(prices), axis=0)` 형태를 지켰는가 |
| 연환산 | 평균×252, 표준편차×√252 를 구분했는가 |
| 표준화 | 종목별 평균과 표준편차로 z-점수를 만들었는가 |
| 시뮬레이션 | 252일×1000회 이상 반복하고 상승 확률을 계산했는가 |
| 결론 | 출력 수치와 모순되지 않는 4줄 결론을 썼는가 |